# Task 2.2: WL Subtree Kernel Implementation

**Paper**: Weisfeiler-Lehman Graph Kernels (Shervashidze et al., JMLR 2011)

**Student**: Meghavi (Roll: 230044)

## Contribution Being Reproduced

We reproduce the **Weisfeiler-Lehman subtree kernel** (Definition 4, Algorithm 2) combined with C-SVM classification (Section 4.2.2).

## Evaluation Metric

**Classification accuracy** via **10-fold cross-validation**, consistent with the paper's evaluation protocol (Section 4.2.2).

In [1]:
import numpy as np
import pickle
import os
from collections import defaultdict
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

SEED = 42
np.random.seed(SEED)

## Load Dataset

In [2]:
data_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'partB', 'data', 'synthetic_dataset.pkl')
with open(data_path, 'rb') as f:
    dataset = pickle.load(f)

graphs = dataset['graphs']
y = dataset['labels']
print(f"Loaded {len(graphs)} graphs, classes: {np.bincount(y)}")

Loaded 200 graphs, classes: [100 100]


## WL Subtree Kernel Implementation

This implements **Algorithm 1** (WL relabeling) and **Definition 4** (WL subtree kernel) from the paper.

The algorithm proceeds as follows:
1. **Initialize**: Use original node labels as iteration-0 labels
2. **For each iteration** $i = 1, \ldots, h$:
   - For each node $v$, collect the multiset of neighbor labels $M_i(v)$ (Algorithm 1, Step 1)
   - Sort $M_i(v)$ and prepend node's own label $l_{i-1}(v)$ (Algorithm 1, Step 2)
   - Compress the resulting string via dictionary-based hash (Algorithm 1, Step 3; Algorithm 2)
   - Update node label $l_i(v) := f(s_i(v))$ (Algorithm 1, Step 4)
3. **Build feature vectors**: Count all labels across iterations 0 through $h$ (Definition 4, Equation 2)

In [3]:
def wl_subtree_features(graphs, h=3, use_label_prefix=True):
    """
    Compute WL subtree kernel feature vectors for a list of graphs.
    
    Implements Algorithm 1 (WL relabeling) and Definition 4 (feature vector
    construction) from Shervashidze et al. (JMLR 2011).
    
    Parameters:
        graphs: list of dicts with 'adj_list', 'node_labels', 'n_nodes'
        h: number of WL iterations (default: 3)
        use_label_prefix: whether to prepend node's own label in Step 2
    
    Returns:
        feature_matrix: scipy sparse matrix of shape (n_graphs, n_features)
        label_map: dictionary mapping label strings to feature indices
    """
    # Build adjacency structures for each graph
    adj_lists = []
    current_labels = []  # List of dicts: node -> label
    
    for g in graphs:
        adj = defaultdict(list)
        for u, v in g['adj_list']:
            adj[u].append(v)
            adj[v].append(u)
        adj_lists.append(adj)
        current_labels.append(dict(g['node_labels']))
    
    # Global label compression dictionary (Algorithm 2: hash function)
    label_counter = 0
    label_lookup = {}  # string -> compressed integer label
    
    # Assign initial compressed labels for original labels
    for g_labels in current_labels:
        for v, lbl in g_labels.items():
            s = str(lbl)
            if s not in label_lookup:
                label_lookup[s] = label_counter
                label_counter += 1
            g_labels[v] = label_lookup[s]
    
    # Collect all label counts: iteration 0 (original labels)
    # feature_counts[graph_idx] = {label_id: count}
    feature_counts = [defaultdict(int) for _ in range(len(graphs))]
    for gi, g_labels in enumerate(current_labels):
        for v, lbl in g_labels.items():
            feature_counts[gi][lbl] += 1
    
    # WL iterations (Algorithm 1)
    for iteration in range(1, h + 1):
        new_labels_all = []
        for gi, (adj, g_labels) in enumerate(zip(adj_lists, current_labels)):
            new_labels = {}
            for v in range(graphs[gi]['n_nodes']):
                # Step 1: Multiset-label determination
                neighbor_labels = sorted([g_labels[u] for u in adj[v]])
                
                # Step 2: String creation (with or without label prefix)
                if use_label_prefix:
                    label_string = str(g_labels[v]) + '_' + '_'.join(map(str, neighbor_labels))
                else:
                    label_string = '_'.join(map(str, neighbor_labels))
                
                # Step 3: Label compression via dictionary hash
                if label_string not in label_lookup:
                    label_lookup[label_string] = label_counter
                    label_counter += 1
                
                # Step 4: Relabeling
                new_labels[v] = label_lookup[label_string]
            
            new_labels_all.append(new_labels)
            
            # Accumulate feature counts for this iteration (Definition 4)
            for v, lbl in new_labels.items():
                feature_counts[gi][lbl] += 1
        
        current_labels = new_labels_all
    
    # Build feature matrix (Equation 2)
    n_features = label_counter
    X = np.zeros((len(graphs), n_features))
    for gi, counts in enumerate(feature_counts):
        for lbl, cnt in counts.items():
            X[gi, lbl] = cnt
    
    return X, label_lookup

print("WL subtree feature function defined.")

WL subtree feature function defined.


## Kernel Matrix Computation

The WL subtree kernel is the **dot product** of feature vectors (Equation 2):
$$K(G, G') = \langle \phi_{\text{WLsubtree}}(G), \phi_{\text{WLsubtree}}(G') \rangle$$

In matrix form: $K = X X^T$

In [4]:
def compute_kernel_matrix(X):
    """Compute kernel matrix K = X @ X.T (dot product kernel)."""
    return X @ X.T

print("Kernel matrix function defined.")

Kernel matrix function defined.


## Classification with SVM

Following Section 4.2.2, we use `SVC(kernel='precomputed')` with 10-fold stratified cross-validation. The regularization parameter $C$ is selected from $\{10^{-3}, 10^{-2}, \ldots, 10^{2}, 10^{3}\}$ based on training fold accuracy.

In [5]:
def classify_with_svm(K, y, n_folds=10, seed=42):
    """
    Classify using SVM with precomputed kernel and 10-fold CV.
    Selects C via nested CV on training folds (Section 4.2.2).
    """
    C_values = [10**i for i in range(-3, 4)]
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_accuracies = []
    
    for train_idx, test_idx in cv.split(np.zeros(len(y)), y):
        K_train = K[np.ix_(train_idx, train_idx)]
        K_test = K[np.ix_(test_idx, train_idx)]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Select best C on training set via inner CV
        best_C = C_values[0]
        best_inner_acc = 0
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        
        for C in C_values:
            svm = SVC(kernel='precomputed', C=C)
            inner_scores = cross_val_score(svm, K_train, y_train, cv=inner_cv, scoring='accuracy')
            if inner_scores.mean() > best_inner_acc:
                best_inner_acc = inner_scores.mean()
                best_C = C
        
        # Train with best C, evaluate on test fold
        svm = SVC(kernel='precomputed', C=best_C)
        svm.fit(K_train, y_train)
        acc = svm.score(K_test, y_test)
        fold_accuracies.append(acc)
    
    return np.mean(fold_accuracies), np.std(fold_accuracies), fold_accuracies

print("SVM classification function defined.")

SVM classification function defined.


## Run the Full Pipeline

### Hyperparameters
- WL iterations $h = 3$ (consistent with paper's typical setting)
- SVM with precomputed kernel
- 10-fold stratified cross-validation
- Random seed: 42

In [6]:
# Compute WL subtree features
h = 3  # Number of WL iterations
X, label_map = wl_subtree_features(graphs, h=h)
print(f"Feature matrix shape: {X.shape}")
print(f"Total unique labels (features): {X.shape[1]}")

# Compute kernel matrix
K = compute_kernel_matrix(X)
print(f"Kernel matrix shape: {K.shape}")

# Classify
mean_acc, std_acc, fold_accs = classify_with_svm(K, y)
print(f"\n{'='*50}")
print(f"WL Subtree Kernel (h={h}) + SVM Classification")
print(f"{'='*50}")
print(f"10-Fold CV Accuracy: {mean_acc*100:.2f}% (+/- {std_acc*100:.2f}%)")
print(f"Per-fold accuracies: {[f'{a*100:.1f}%' for a in fold_accs]}")

Feature matrix shape: (200, 3161)
Total unique labels (features): 3161
Kernel matrix shape: (200, 200)



WL Subtree Kernel (h=3) + SVM Classification
10-Fold CV Accuracy: 88.00% (+/- 5.57%)
Per-fold accuracies: ['85.0%', '95.0%', '85.0%', '95.0%', '80.0%', '90.0%', '80.0%', '95.0%', '85.0%', '90.0%']


In [7]:
# Save results for use in task_2_3
results = {
    'h': h,
    'mean_acc': mean_acc,
    'std_acc': std_acc,
    'fold_accs': fold_accs,
    'n_features': X.shape[1],
    'feature_matrix': X,
    'kernel_matrix': K
}

results_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'partB', 'results')
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, 'wl_results.pkl'), 'wb') as f:
    pickle.dump(results, f)
print("Results saved.")

Results saved.
